# Non-LLM baseline: TF-IDF retrieval and SRS evidence

This notebook runs `src/compliance_copilot/` on 14 SRS-addressable NASA clauses
and 54 requirements. See [baseline documentation](../docs/baseline.md) for the
Mermaid workflow, rules and benchmark schema. The original notebook retains preparation work.

Install `requirements.txt` first. In Colab, clone branch
`baseline/issue-2-non-llm-classifier` and work inside the cloned repository.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/compliance_copilot").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from compliance_copilot.pipeline import run_baseline


## Run

TF-IDF learns word weights from clause text, transforms requirements using those
weights, and ranks clauses by cosine similarity. Up to three positive matches
are returned; no shared indexed words produces `NO_LEXICAL_MATCH`.

Scores measure wording similarity. Evidence rules assess named SRS provisions
afterward; uncertain or unsupported pairs return `REVIEW`. Document checks run
separately so missing evidence does not depend on retrieval success.


In [ ]:
result = run_baseline(ROOT, top_k=3)
display(result["metrics"])


## Inspect keyword findings and candidate clauses

Keyword checks flag ambiguity; `NO_RULE_MATCH` does not establish compliance.
Inspect security-related examples against the clause text. Some have no match
because TF-IDF does not understand synonyms or word forms.


In [ ]:
display(result["keywords"].loc[
    result["keywords"]["status"] == "NEEDS_REVIEW",
    ["requirement_id", "title", "findings"],
].head(10))
display(result["matches"].loc[
    result["matches"]["requirement_id"].isin([
        "SFMC-REQ-048", "SFMC-REQ-049", "SFMC-REQ-050", "SFMC-REQ-054"
    ]),
    ["requirement_id", "swe_id", "clause_id", "similarity_score", "retrieval_status", "clause_text"],
])


## Inspect document evidence

`DOCUMENT` results search beyond individual sentences. A list of mission needs
does not supply requirement-to-parent links. Verdicts apply only to the listed
`assessed_provision`, not full NASA compliance. Unsupported provisions and
external records require review.


In [ ]:
display(result["document_assessments"][[
    "swe_id", "clause_id", "assessed_provision", "verdict", "evidence", "missing_evidence"
]])


## Evaluate when advisor labels arrive

No benchmark or Data Card is committed. Once the advisor confirms label scope
matches the documented provisions, use:

```python
evaluated = run_baseline(ROOT, benchmark_path=ROOT / "data/benchmark.csv", split="test")
display(evaluated["metrics"])
```

Omit `split` if there is no partition column. Classification uses labeled pairs
directly; retrieval is evaluated separately. Reviews count toward coverage and
missed gaps. Unit-test fixtures are not a performance benchmark.


## Exported results

CSV and JSON files are written to `output/baseline/`: keyword findings, candidate
matches, pair assessments, document assessments and `metrics.json`. JSON retains
evidence lists. Outputs are generated locally and ignored by Git. Re-run
`python scripts/prepare_data.py` after source changes.
